# PDF Report Pipeline

fpdf2 layout engine, plotly-to-PNG conversion, bilingual commentary templates, and HTTP response strategy.

## 1. Report Config Schema

Parameters the frontend sends to `GET /api/v1/report/generate`.

In [ ]:
# Report request parameters (query string)
report_config = {
    "segment":     None,            # Optional[str]: filter to a customer segment
    "start_month": "2024-01",      # Optional[str]: inclusive date range start
    "end_month":   "2025-12",      # Optional[str]: inclusive date range end
    "lang":        "en",           # str: 'en' or 'es'
    "sections":    "cover,executive_summary,kpi_dashboard,revenue_analysis,customer_health,anomaly_alerts,forecast,recommendations",
    # Comma-separated section names; None = all sections
}

all_sections = [
    "cover",
    "executive_summary",
    "kpi_dashboard",
    "revenue_analysis",
    "customer_health",
    "anomaly_alerts",
    "forecast",
    "recommendations",
]

# Parse sections from comma-separated string
def parse_sections(sections_param):
    if sections_param is None:
        return all_sections
    return [s.strip() for s in sections_param.split(",") if s.strip()]

active = parse_sections(report_config["sections"])
print(f"Active sections ({len(active)}): {active}")

## 2. Plotly to PNG

kaleido renders plotly figures server-side. Temporary files are used to pass image data to fpdf2's `image()` method, which requires a file path.

In [ ]:
import tempfile
import os
from typing import Optional

import plotly.graph_objects as go


def render_chart_png(fig: go.Figure) -> Optional[str]:
    """Render a plotly Figure to a temporary PNG file.

    Returns temp file path or None if kaleido is unavailable.
    width=700, height=350, scale=2 produces 1400x700 px at 144 dpi -- sharp in PDF.
    """
    try:
        tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
        fig.write_image(tmp.name, width=700, height=350, scale=2)
        return tmp.name
    except Exception as e:
        print(f"  Chart render failed: {e}")
        return None


# Demonstration: create a simple MRR sparkline
import pandas as pd

monthly = pd.read_parquet("data/processed/monthly_metrics.parquet")
monthly["month"] = pd.to_datetime(monthly["month"])
monthly = monthly.sort_values("month")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=monthly["month"].dt.strftime("%Y-%m"),
    y=monthly["mrr"],
    mode="lines",
    line=dict(color="#2563eb", width=2),
    fill="tozeroy",
    fillcolor="rgba(37,99,235,0.1)",
))
fig.update_layout(
    title="MRR Trend",
    title_font_size=14,
    margin=dict(l=40, r=20, t=40, b=30),
    height=250,
    width=700,
    showlegend=False,
    plot_bgcolor="white",
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridcolor="#f0f0f0")

path = render_chart_png(fig)
if path:
    size_kb = os.path.getsize(path) / 1024
    print(f"Chart saved to: {path}  ({size_kb:.1f} KB)")
    os.unlink(path)  # cleanup
else:
    print("kaleido not available -- chart rendering skipped")

## 3. fpdf2 Layout Engine

Page setup, color palette constants, and the custom `_KPIReport` subclass.

In [ ]:
from fpdf import FPDF
from datetime import datetime

# Color palette (RGB)
COLOR_BRAND_BLUE  = (37, 99, 235)   # primary blue
COLOR_TEXT_DARK   = (30, 30, 30)    # section titles
COLOR_TEXT_BODY   = (50, 50, 50)    # body text
COLOR_TEXT_MUTED  = (100, 100, 100) # header/footer
COLOR_RULE_LIGHT  = (200, 200, 200) # separator lines
COLOR_ROW_STRIPE  = (245, 247, 250) # alternating table row
COLOR_STATUS_GREEN  = (34, 197, 94)
COLOR_STATUS_YELLOW = (234, 179, 8)
COLOR_STATUS_RED    = (239, 68, 68)

# Page geometry
PAGE_MARGIN = 10   # mm, all sides
CONTENT_W   = 190  # mm = A4 width (210) minus 2*10 margin


class KPIReport(FPDF):
    """Custom FPDF subclass with header, footer, and shared formatting helpers."""

    def __init__(self, lang: str = "en"):
        super().__init__()
        self.lang = lang
        self.set_auto_page_break(auto=True, margin=20)  # 20mm bottom margin before auto page break

    def header(self):
        self.set_font("Helvetica", "B", 10)
        self.set_text_color(*COLOR_TEXT_MUTED)
        title = "Reporte Ejecutivo de KPIs" if self.lang == "es" else "Executive KPI Report"
        self.cell(0, 8, title, align="L")
        self.cell(0, 8, datetime.now().strftime("%Y-%m-%d"), align="R", new_x="LMARGIN", new_y="NEXT")
        self.set_draw_color(*COLOR_RULE_LIGHT)
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(4)

    def footer(self):
        self.set_y(-15)
        self.set_font("Helvetica", "I", 8)
        self.set_text_color(*COLOR_TEXT_MUTED)
        label = f"Pagina {self.page_no()}/{{nb}}" if self.lang == "es" else f"Page {self.page_no()}/{{nb}}"
        self.cell(0, 10, label, align="C")
        # {nb} is an alias for total page count -- replaced by alias_nb_pages() at output time

    def section_title(self, title: str):
        self.set_font("Helvetica", "B", 14)
        self.set_text_color(*COLOR_TEXT_DARK)
        self.ln(6)
        self.cell(0, 10, title, new_x="LMARGIN", new_y="NEXT")
        self.set_draw_color(*COLOR_BRAND_BLUE)
        self.set_line_width(0.5)
        self.line(10, self.get_y(), 80, self.get_y())  # 70mm accent underline
        self.set_line_width(0.2)
        self.ln(4)

    def body_text(self, text: str):
        self.set_font("Helvetica", "", 10)
        self.set_text_color(*COLOR_TEXT_BODY)
        self.multi_cell(0, 6, text)  # wraps automatically at page width
        self.ln(2)


# Test the layout
pdf = KPIReport(lang="en")
pdf.alias_nb_pages()
pdf.add_page()
pdf.section_title("Test Section")
pdf.body_text("This is a test of the body text layout with automatic word wrapping.")

test_bytes = bytes(pdf.output())
print(f"Test PDF generated: {len(test_bytes):,} bytes")

## 4. KPI Table Rendering

Column widths sum to 190mm (CONTENT_W). Status column uses traffic light colors.

In [ ]:
# Column layout (mm): [KPI, Value, MoM%, Target, YoY%, Status, Category]
COL_WIDTHS = [40, 30, 25, 30, 25, 20, 20]  # sum = 190mm
assert sum(COL_WIDTHS) == 190, "Column widths must sum to page content width"

STATUS_COLORS = {
    "green":  (34, 197, 94),
    "yellow": (234, 179, 8),
    "red":    (239, 68, 68),
}


def render_kpi_table(pdf: KPIReport, kpis: list):
    """Render the KPI table with alternating row stripes and traffic light status colors."""
    # Header row
    pdf.set_font("Helvetica", "B", 9)
    pdf.set_fill_color(*COLOR_BRAND_BLUE)
    pdf.set_text_color(255, 255, 255)

    headers_en = ["KPI", "Value", "MoM %", "Target", "YoY %", "Status", "Category"]
    headers_es = ["KPI", "Valor", "MoM %", "Meta",  "YoY %", "Estado", "Categoria"]
    headers = headers_es if pdf.lang == "es" else headers_en

    for i, h in enumerate(headers):
        pdf.cell(COL_WIDTHS[i], 7, h, border=1, fill=True, align="C")
    pdf.ln()

    # Data rows
    pdf.set_font("Helvetica", "", 8)
    pdf.set_text_color(*COLOR_TEXT_BODY)

    for j, kpi in enumerate(kpis):
        fill = j % 2 == 0
        if fill:
            pdf.set_fill_color(*COLOR_ROW_STRIPE)

        status = kpi.get("traffic_light", "green")
        row_data = [
            str(kpi.get("name", "")),
            str(kpi.get("formatted", kpi.get("value", ""))),
            f"{kpi.get('change_mom', 0) * 100:+.1f}%",
            str(kpi.get("target", "")),
            f"{kpi.get('change_yoy', 0) * 100:+.1f}%",
            status.upper(),
            str(kpi.get("category", "")),
        ]

        for i, val in enumerate(row_data):
            if i == 5:  # status column: apply traffic light color
                r, g, b = STATUS_COLORS.get(status, (100, 100, 100))
                pdf.set_text_color(r, g, b)
                pdf.set_font("Helvetica", "B", 8)
            pdf.cell(COL_WIDTHS[i], 6, val, border=1, fill=fill, align="C")
            if i == 5:
                pdf.set_text_color(*COLOR_TEXT_BODY)
                pdf.set_font("Helvetica", "", 8)
        pdf.ln()


# Smoke test with 3 dummy KPIs
dummy_kpis = [
    {"name": "MRR",       "formatted": "$950K",  "change_mom": 0.03,  "target": 950000, "change_yoy": 0.35, "traffic_light": "green",  "category": "revenue"},
    {"name": "Logo Churn","formatted": "2.5%",   "change_mom": 0.001, "target": 0.02,   "change_yoy": 0.05, "traffic_light": "yellow", "category": "customers"},
    {"name": "NPS",        "formatted": "38",     "change_mom": -0.02, "target": 55,     "change_yoy": -0.1, "traffic_light": "red",    "category": "customers"},
]

pdf2 = KPIReport(lang="en")
pdf2.alias_nb_pages()
pdf2.add_page()
pdf2.section_title("KPI Dashboard")
render_kpi_table(pdf2, dummy_kpis)

test2 = bytes(pdf2.output())
print(f"KPI table PDF: {len(test2):,} bytes")
print(f"Column widths: {COL_WIDTHS} -> total {sum(COL_WIDTHS)}mm")

## 5. Chart Page Insertion

Images are placed at x=10 (left margin), width=190mm (full content width). fpdf2 preserves aspect ratio automatically when only width is specified.

In [ ]:
# MRR waterfall chart
def create_waterfall_chart(waterfall: dict):
    categories = ["Starting MRR", "New", "Expansion", "Contraction", "Churned", "Ending MRR"]
    values = [
        waterfall["starting_mrr"],
        waterfall["new"],
        waterfall["expansion"],
        -waterfall["contraction"],
        -waterfall["churned"],
        waterfall["ending_mrr"],
    ]
    measures = ["absolute", "relative", "relative", "relative", "relative", "total"]

    fig = go.Figure(go.Waterfall(
        x=categories,
        y=values,
        measure=measures,
        connector=dict(line=dict(color="#d1d5db")),
        increasing=dict(marker=dict(color="#22c55e")),
        decreasing=dict(marker=dict(color="#ef4444")),
        totals=dict(marker=dict(color="#2563eb")),
    ))
    fig.update_layout(
        title="MRR Waterfall",
        title_font_size=14,
        margin=dict(l=40, r=20, t=40, b=30),
        height=300,
        width=700,
        plot_bgcolor="white",
    )
    return fig


# Image insertion pattern in generate_report():
#   path = render_chart_png(fig)         # temp PNG path
#   if path:
#       temp_files.append(path)          # track for cleanup
#       pdf.image(path, x=10, w=190)    # x=left margin, w=full content width
#                                        # height auto-computed to preserve aspect ratio

print("Chart insertion pattern:")
print("  pdf.image(path, x=10, w=190)")
print("  -> places image at left margin, spanning full 190mm content width")
print("  -> fpdf2 auto-scales height to maintain original aspect ratio")
print("  -> kaleido renders at scale=2 so image is 1400px wide (retina quality)")

## 6. Bilingual Commentary

Template-based NLG (natural language generation) using f-string templates with conditional logic. No Jinja2 in production -- the module uses plain Python string formatting.

In [ ]:
def generate_executive_summary(metrics: dict, lang: str = "en") -> str:
    """3-5 sentence executive summary built from structured KPI data."""
    health = metrics.get("health_score", 0)
    mrr    = metrics.get("mrr", 0)
    mrr_g  = metrics.get("mrr_growth_rate", 0)
    nrr    = metrics.get("nrr", metrics.get("net_revenue_retention", 0))
    churn  = metrics.get("logo_churn_rate", 0)
    nps    = metrics.get("nps", 0)

    def fmt_currency(v):
        if abs(v) >= 1_000_000: return f"${v/1_000_000:,.1f}M"
        if abs(v) >= 1_000:     return f"${v/1_000:,.0f}K"
        return f"${v:,.0f}"

    def fmt_pct(v):
        return f"{v*100:.1f}%" if abs(v) <= 1 else f"{v:.1f}%"

    if health >= 75:   tone_en, tone_es = "strong", "solida"
    elif health >= 50: tone_en, tone_es = "moderate", "moderada"
    else:              tone_en, tone_es = "concerning", "preocupante"

    # Conditional MRR direction sentence
    if mrr_g > 0.02:
        mrr_en = f"MRR grew {fmt_pct(mrr_g)} MoM to {fmt_currency(mrr)}, signaling healthy momentum."
        mrr_es = f"El MRR crecio {fmt_pct(mrr_g)} mes a mes hasta {fmt_currency(mrr)}."
    elif mrr_g >= 0:
        mrr_en = f"MRR reached {fmt_currency(mrr)} with {fmt_pct(mrr_g)} growth, indicating a stable but slowing trajectory."
        mrr_es = f"El MRR alcanzo {fmt_currency(mrr)} con crecimiento de {fmt_pct(mrr_g)}."
    else:
        mrr_en = f"MRR declined {fmt_pct(mrr_g)} to {fmt_currency(mrr)}, requiring immediate attention."
        mrr_es = f"El MRR disminuyo {fmt_pct(mrr_g)} a {fmt_currency(mrr)}, requiere atencion inmediata."

    if lang == "es":
        return (
            f"La salud general de NovaCRM se evalua como {tone_es} con un puntaje de {health:.1f}/100. "
            f"{mrr_es} "
            f"La tasa de abandono se ubica en {fmt_pct(churn)}. "
            f"El NPS es de {nps:.0f}."
        )
    return (
        f"NovaCRM's overall health score stands at {health:.1f}/100, reflecting a {tone_en} position. "
        f"{mrr_en} "
        f"Logo churn stands at {fmt_pct(churn)}. "
        f"NPS of {nps:.0f}."
    )


# Test with realistic values
sample_metrics = {
    "health_score": 67.3,
    "mrr": 923_000,
    "mrr_growth_rate": 0.024,
    "nrr": 1.08,
    "logo_churn_rate": 0.028,
    "nps": 42,
}

print("English:")
print(generate_executive_summary(sample_metrics, lang="en"))
print("\nSpanish:")
print(generate_executive_summary(sample_metrics, lang="es"))

## 7. PDF Bytes to HTTP Response

The report router returns a `StreamingResponse` with PDF bytes. This avoids writing to a temp file that would need cleanup coordination across concurrent requests.

In [ ]:
import io

# In the router (backend/kpi_backend/routers/report.py):
#
#   pdf_bytes = generate_report(report_data, lang=lang, sections=section_list)
#
#   return StreamingResponse(
#       io.BytesIO(pdf_bytes),
#       media_type="application/pdf",
#       headers={"Content-Disposition": f'attachment; filename="{filename}"'},
#   )
#
# The frontend (ReportPanel.tsx) calls this endpoint and triggers a browser download:
#   const blob = await response.blob()
#   const url = URL.createObjectURL(blob)
#   a.href = url; a.download = filename; a.click()

# Temp file cleanup pattern (used for chart PNGs):
#   temp_files: List[str] = []
#   try:
#       # ... build PDF, accumulating temp file paths in temp_files ...
#       return bytes(pdf.output())
#   finally:
#       for f in temp_files:
#           try:
#               os.unlink(f)
#           except OSError:
#               pass  # ignore errors if file already removed

# Simulate the output contract
pdf_demo = KPIReport(lang="en")
pdf_demo.alias_nb_pages()
pdf_demo.add_page()
pdf_demo.section_title("Executive Summary")
pdf_demo.body_text(generate_executive_summary(sample_metrics, lang="en"))

pdf_bytes = bytes(pdf_demo.output())
pdf_stream = io.BytesIO(pdf_bytes)

print(f"PDF size:          {len(pdf_bytes):,} bytes")
print(f"BytesIO readable:  {pdf_stream.readable()}")
print(f"Content-Type:      application/pdf")
print(f"Disposition:       attachment; filename=kpi_report_2025-12_en.pdf")
print()
print("StreamingResponse wraps the BytesIO object -- fpdf2 output is already in memory,")
print("so streaming has no latency benefit here, but it follows FastAPI best practice")
print("for binary responses (avoids JSON serialization overhead).")